In [2]:
import os
import json
import time
import re
import random
from datetime import datetime
from typing import List, Dict, Any, Generator
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pinecone import Pinecone, ServerlessSpec
import fitz  # PyMuPDF
import docx2txt  # Word extraction
import serpapi
import langgraph
import langchain
import serpapi
import wikipedia

In [3]:
# ---------------- LOAD ENV VARIABLES ----------------
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
secret_api_key = os.getenv("SERPAPI_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = os.getenv("INDEX_NAME", "studybuddy")
PINECONE_ENVIRONMENT = os.getenv("PINECONE_ENVIRONMENT", "us-east1-gcp")


In [24]:

if not PINECONE_API_KEY:
    raise ValueError("❌ PINECONE_API_KEY not found!")

#---------------- CONFIGURE GEMINI & PINECONE ----------------
#Initialize Gemini client with latest google-genai SDK
client = genai.Client(api_key=GEMINI_API_KEY)

# Initialize Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

# Create or connect to Pinecone index
existing_indexes = [i.name for i in pc.list_indexes()]
if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region=PINECONE_ENVIRONMENT)
    )
index = pc.Index(INDEX_NAME)


In [ ]:
from langchain_core.tools import tool
@tool
def search_wikipedia(query: str) -> str:
 """Searches Wikipedia for the given query and returns a summary of the results."""
 result = wikipedia.search(query=query)
 summary = wikipedia.summary(result, sentences=15)
 return summary

In [34]:
tools = [search_wikipedia]

In [35]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY")
)
llm_with_tools = llm.bind_tools(tools)

In [36]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# 1. State definition
class State(TypedDict):
    messages: Annotated[list, add_messages]

# 2. Agent Node (Invokes LLM with bound tools)
def call_model(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# 3. Tool Node (Executes tools when LLM returns tool_calls)
tool_node = ToolNode(tools)


In [37]:
from langgraph.graph import END

def route_after_model(state: State):
    last_message = state["messages"][-1]
    
    # If the LLM requested a tool call, route to 'tools' node
    if last_message.tool_calls:
        return "tools"
    
    # If no tool calls were requested, stop and return to the user
    return END


In [38]:
from langgraph.graph import StateGraph, START

workflow = StateGraph(State)

# Add our processing units
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

# Connect flow logic
workflow.add_edge(START, "agent")

# Add conditional routing after the agent node finishes
workflow.add_conditional_edges(
    "agent",
    route_after_model,
    {
        "tools": "tools",  # routes to 'tools' node if route_after_model returns 'tools'
        END: END          # routes to END if route_after_model returns END
    }
)

# Crucial: route back to the agent after tool execution so it can read tool outputs!
workflow.add_edge("tools", "agent")

# Compile into an executable application
app = workflow.compile()


In [39]:
# Scenario A: Triggers 'get_user_balance' tool automatically
input_state = {"messages": [("user", "Who is  Abrahim Lincoln?")]}
for chunk in app.stream(input_state):
    print(chunk)



{'agent': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_wikipedia', 'arguments': '{"query": "Abraham Lincoln"}'}, '__gemini_function_call_thought_signatures__': {'ac5978c3-dde2-4c91-9505-b1c6e8735c1a': 'CsUDAQw51scAqLLUBdcmlM/1maS3BpOd+aF9Exbi9vgoNNwmOP8lmEM7+PqmMIMQ9JaSAwdxVcNWLwg73tSpLJUhcuXlgrnL/thj6CAK3plRmeYUWKkcQXD/bqI2e+az/ULVsLS94er0NU6Jl0WZjnXee1PKYNt/lpdtbQGvgYQBBNe5HyT8kiyb8ENPssjjfdsOsa0ZOKS21PUPKqn0gDxTPAdgP7pdeACSEAkpXMSP0RW6PBR8wUhiu3hX+TCPhBzir7OHsy9Iusz+5d6QigSeh1WaadXKmKsERIEH4q+ROQmdp3QASAjlY3MUCS1hX4xjv/8XWHfdWqqo93LTnGEnvlFBzW9ThB+n6ABUVxtK3MQJb7Qo34eRQK8fNvP6gPe54nsCk5X2ZzTmXONlQu7TMheDGnWZbZZIU31BbGFMwwNV3qyqGeuc7wyLjEYxcmTkGj9xX2F8+fRFRx6L8uhDXevWAA0kg4zgCXxDNSwrbgGT8TcHPn1Kszs4PnGN4rM109uumehGrOGowzQ1gQ0vZHhowIAqRHNffZTmKryp5bJnI3ztWveo0vm2iUUm5nRYuOae4Qo94NApD4X+wsgckkpnUWrC'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc

JSONDecodeError: Expecting value: line 1 column 1 (char 0)